# Bonus: SQL (adaptado a lab local)

**Qué se ve aquí:** un agente que consulta una base SQLite (`resources/Chinook.db`, tienda de música) escribiendo SQL con una tool.

**Convención de este fork:** 🔸 ORIGINAL DEL CURSO (comentada, con el motivo) → 🟢 ADAPTADO LOCAL (la que corre con Ollama). Celdas sin marca = iguales al curso.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: falta cargar local_model.py (raíz del repo) y definir el modelo local.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from dotenv import load_dotenv
#
# load_dotenv()

In [ ]:
# 🟢 ADAPTADO LOCAL
# Motivo: el notebook está en notebooks/module-2/ y local_model.py en la raíz del repo.
#         MODEL reemplaza a "gpt-5-nano" en todas las celdas (gemma4: soporta tools, el más rápido del lab).
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "local_model.py").exists())))

from dotenv import load_dotenv
load_dotenv()
from local_model import get_model, OLLAMA_URL

MODEL = get_model("gemma4:latest", num_ctx=16384)   # el esquema de la BD + resultados SQL ocupan contexto

In [ ]:
# Sin cambios
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

In [ ]:
# Sin cambios
from langchain.tools import tool

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

sql_query.invoke("SELECT * FROM Artist LIMIT 10")

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents import create_agent
#
# agent = create_agent(
#     model="gpt-5-nano",
#     tools=[sql_query]
# )

In [ ]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" → MODEL (gemma4 en Ollama, definido en la celda de setup).
from langchain.agents import create_agent

agent = create_agent(
    model=MODEL,
    tools=[sql_query]
)

In [ ]:
# Sin cambios
from langchain.messages import HumanMessage

question = HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?")

response = agent.invoke(
    {"messages": [question]}
)

In [ ]:
# Sin cambios
from pprint import pprint

pprint(response['messages'])

In [ ]:
# Sin cambios
print(response["messages"][-3].tool_calls[0]['args']['query'])